In [1]:
!pip install wandb thop

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
import wandb
from thop import profile

# Weights & Biases Setup
hyperparameters = {
    "learning_rate": 0.001,
    "epochs": 25,
    "batch_size": 128,
    "optimizer": "Adam",
    "model_architecture": "ResNet18",
    "dataset": "CIFAR-10"
}

wandb.init(project="CIFAR10_Assignment_Lab2", config=hyperparameters)


# Custom Dataset & Dataloader
class CustomCIFAR10Dataset(Dataset):
    def __init__(self, train=True, transform=None):
        self.dataset = torchvision.datasets.CIFAR10(
            root='./data', train=train, download=True
        )
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Normalization parameters for CIFAR-10
mean = (0.4914, 0.4822, 0.4465)
std = (0.2023, 0.1994, 0.2010)

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_set = CustomCIFAR10Dataset(train=True, transform=transform_train)
test_set = CustomCIFAR10Dataset(train=False, transform=transform_test)

train_loader = DataLoader(train_set, batch_size=hyperparameters["batch_size"], shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=hyperparameters["batch_size"], shuffle=False, num_workers=2)





/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: b22bb038 (b22bb038-indian-institute-of-technology-jodhpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 170M/170M [00:05<00:00, 31.1MB/s]


In [3]:
# Model Selection & FLOPs Counting
model = torchvision.models.resnet18(num_classes=10)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Count FLOPs using a dummy input to log static model analysis
dummy_input = torch.randn(1, 3, 32, 32).to(device)
flops, params = profile(model, inputs=(dummy_input, ), verbose=False)

print(f"Total FLOPs: {flops / 1e6:.2f} Million")
print(f"Total Parameters: {params / 1e6:.2f} Million")

# Log complexity once to the metrics group
wandb.log({
    "metrics/total_gflops": flops / 1e9,
    "metrics/total_params_M": params / 1e6
})

# Visualization Utilities
def log_grad_flow(named_parameters):
    ave_grads = []
    layers = []
    for n, p in named_parameters:
        if(p.requires_grad) and ("bias" not in n):
            layers.append(n)
            if p.grad is not None:
                ave_grads.append(p.grad.abs().mean().cpu().item())
            else:
                ave_grads.append(0)

    data = [[label, val] for (label, val) in zip(layers, ave_grads)]
    table = wandb.Table(data=data, columns=["Layer", "Avg Gradient"])
    wandb.log({"gradient_flow_chart": wandb.plot.bar(table, "Layer", "Avg Gradient", title="Gradient Flow")})


# Training Loop with Grouped Logging
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparameters["learning_rate"])


Total FLOPs: 37.22 Million
Total Parameters: 11.18 Million


In [4]:
# Watch model for histograms of weights and gradients automatically
wandb.watch(model, log="all", log_freq=100)

for epoch in range(hyperparameters["epochs"]):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    # Store weights before update for "Weight Update Flow" analysis
    weights_before = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}

    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        if i % 100 == 0:
            log_grad_flow(model.named_parameters())

        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    # Calculate Update Flow Magnitude
    update_mags = [torch.norm(p.detach() - weights_before[n]).cpu().item()
                   for n, p in model.named_parameters() if p.requires_grad]

    # Validation
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    wandb.log({
        "epoch": epoch,
        "train/loss": running_loss / len(train_loader),
        "train/acc": 100. * correct / total,
        "val/loss": val_loss / len(test_loader),
        "val/acc": 100. * val_correct / val_total,
        "metrics/avg_weight_update_mag": np.mean(update_mags)
    })

    print(f"Epoch {epoch+1}/{hyperparameters['epochs']} | Train Acc: {100.*correct/total:.2f}% | Val Acc: {100.*val_correct/val_total:.2f}%")

print("Training Complete.")
wandb.finish()

Epoch 1/25 | Train Acc: 43.70% | Val Acc: 52.79%
Epoch 2/25 | Train Acc: 57.53% | Val Acc: 61.63%
Epoch 3/25 | Train Acc: 63.98% | Val Acc: 65.20%
Epoch 4/25 | Train Acc: 67.64% | Val Acc: 69.00%
Epoch 5/25 | Train Acc: 70.35% | Val Acc: 71.89%
Epoch 6/25 | Train Acc: 72.46% | Val Acc: 72.91%
Epoch 7/25 | Train Acc: 74.05% | Val Acc: 74.25%
Epoch 8/25 | Train Acc: 75.65% | Val Acc: 76.02%
Epoch 9/25 | Train Acc: 76.49% | Val Acc: 73.63%
Epoch 10/25 | Train Acc: 78.01% | Val Acc: 77.95%
Epoch 11/25 | Train Acc: 78.51% | Val Acc: 77.41%
Epoch 12/25 | Train Acc: 79.59% | Val Acc: 78.07%
Epoch 13/25 | Train Acc: 80.21% | Val Acc: 77.85%
Epoch 14/25 | Train Acc: 80.80% | Val Acc: 79.09%
Epoch 15/25 | Train Acc: 81.48% | Val Acc: 80.49%
Epoch 16/25 | Train Acc: 82.19% | Val Acc: 80.46%
Epoch 17/25 | Train Acc: 82.59% | Val Acc: 80.63%
Epoch 18/25 | Train Acc: 83.35% | Val Acc: 80.52%
Epoch 19/25 | Train Acc: 83.57% | Val Acc: 80.25%
Epoch 20/25 | Train Acc: 84.01% | Val Acc: 81.11%
Epoch 21/

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
metrics/avg_weight_update_mag,▅▁▁▁▂▂▂▂▃▃▄▄▄▅▅▅▆▆▆▇▇▇▇██
metrics/total_gflops,▁
metrics/total_params_M,▁
train/acc,▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇████████
train/loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val/acc,▁▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇████
val/loss,█▆▅▄▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁
epoch,24
metrics/avg_weight_update_mag,1.7004
metrics/total_gflops,0.03722
